# 発展：主成分分析（PCA）

ここでは、第13回で扱った**最小二乗法と相関**を少し発展させて、主成分分析（Principal Component Analysis; PCA）の考え方を学びます。

主成分分析は、多数の変数を含むデータについて、

- どのような変化がデータ全体に共通しているか
- データのばらつきを少数の特徴で表せないか

を調べるためによく使われる方法です。

このNotebookは**発展内容**です。Codeセルは基本的に上から順番に実行してください。

## 1. 2つの変数の散布図から考える

まず、2つの変数 $x$, $y$ が一緒に変化する簡単なデータを作ります。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
x = np.linspace(-3, 3, 40)
y = 1.5*x + rng.normal(0, 1.0, len(x))

plt.scatter(x, y)
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()

散布図を見ると、点は完全に一直線ではありませんが、ある方向に細長く分布しています。

第13回の回帰分析では、

$$
y=ax+b
$$

という直線を考え、**$x$ を固定したときの $y$ 方向の残差**を小さくしました。

一方、主成分分析では発想が少し異なります。

> **データが最も大きく広がっている方向はどちらか**

を探します。

## 2. まず平均を引く

PCAでは、各変数から平均を引き、データの中心を原点へ移します。

In [ ]:
X = np.column_stack([x, y])
Xc = X - X.mean(axis=0)

plt.scatter(Xc[:,0], Xc[:,1])
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("x - mean(x)")
plt.ylabel("y - mean(y)")
plt.axis("equal")
plt.show()

print("平均 =", Xc.mean(axis=0))

平均を引いても、点どうしの位置関係や、散布図が伸びている方向は変わりません。

これで、データの中心を通る方向だけを考えればよくなります。

## 3. ある方向からデータを見る

単位ベクトル

$$
\mathbf{u}
=
\begin{pmatrix}
u_x\\
u_y
\end{pmatrix}
$$

で表される方向を考えます。

各データ

$$
\mathbf{x}_i=
\begin{pmatrix}
x_i\\
y_i
\end{pmatrix}
$$

をその方向へ写した値は、

$$
z_i=\mathbf{x}_i\cdot\mathbf{u}
$$

です。

この $z_i$ のばらつきが大きい方向ほど、その方向にデータが大きく広がっていることになります。

実際に方向を少しずつ変えて、その方向に写したデータの分散を計算してみます。

In [ ]:
angles = np.linspace(0, 180, 361)
variances = []

for angle in angles:
    theta = np.deg2rad(angle)
    u = np.array([np.cos(theta), np.sin(theta)])
    z = Xc @ u
    variances.append(np.mean(z**2))

variances = np.array(variances)

plt.plot(angles, variances)
plt.xlabel("Direction (degree)")
plt.ylabel("Variance along the direction")
plt.show()

best_angle = angles[np.argmax(variances)]
print("最も分散が大きい方向 =", best_angle, "degree")

分散は方向によって変わり、ある方向で最大になります。

**この「データのばらつきが最大になる方向」が第1主成分です。**

In [ ]:
theta = np.deg2rad(best_angle)
u1_trial = np.array([np.cos(theta), np.sin(theta)])

scale = 5
plt.scatter(Xc[:,0], Xc[:,1])
plt.plot([-scale*u1_trial[0], scale*u1_trial[0]],
         [-scale*u1_trial[1], scale*u1_trial[1]],
         linewidth=2, label="1st principal direction")
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("x - mean(x)")
plt.ylabel("y - mean(y)")
plt.axis("equal")
plt.legend()
plt.show()

## 4. 共分散行列

2変数のばらつきと、一緒に変化する程度は、**共分散行列**にまとめることができます。

平均を引いたデータについて、

$$
\mathbf{C}
=
\frac{1}{N}
\begin{pmatrix}
\sum x_i^2 & \sum x_i y_i\\
\sum x_i y_i & \sum y_i^2
\end{pmatrix}
$$

とします。

対角成分はそれぞれ $x$, $y$ の分散、対角以外の成分は $x$ と $y$ の共分散です。

In [ ]:
C = (Xc.T @ Xc) / len(Xc)
print(C)

ある方向 $\mathbf{u}$ に写したデータの分散は、

$$
\frac{1}{N}\sum_i z_i^2
=
\mathbf{u}^{\mathsf T}\mathbf{C}\mathbf{u}
$$

と書くことができます。

したがってPCAは、長さ1の $\mathbf{u}$ の中から

$$
\mathbf{u}^{\mathsf T}\mathbf{C}\mathbf{u}
$$

が最大になる方向を探していることになります。

## 5. 固有値・固有ベクトルで主成分を求める

この最大となる方向は、共分散行列の**固有ベクトル**として求めることができます。

対応する**固有値**は、その主成分方向の分散を表します。

In [ ]:
eigvals, eigvecs = np.linalg.eigh(C)

order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

print("固有値 =", eigvals)
print("第1主成分の方向 =", eigvecs[:,0])
print("第2主成分の方向 =", eigvecs[:,1])

第1主成分と第2主成分を散布図に重ねてみます。

In [ ]:
u1 = eigvecs[:,0]
u2 = eigvecs[:,1]

scale1 = 2*np.sqrt(eigvals[0])
scale2 = 2*np.sqrt(eigvals[1])

plt.scatter(Xc[:,0], Xc[:,1])
plt.plot([-scale1*u1[0], scale1*u1[0]],
         [-scale1*u1[1], scale1*u1[1]],
         linewidth=2, label="PC1")
plt.plot([-scale2*u2[0], scale2*u2[0]],
         [-scale2*u2[1], scale2*u2[1]],
         linewidth=2, label="PC2")
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("x - mean(x)")
plt.ylabel("y - mean(y)")
plt.axis("equal")
plt.legend()
plt.show()

第1主成分はデータが最も大きく広がる方向、第2主成分はそれと直交する方向です。

第13回の回帰直線と見た目は似ていますが、考えているものは同じではありません。

- **回帰分析**：$x$ を説明変数として、$y$ 方向の残差を小さくする
- **PCA**：$x$ と $y$ を対等に扱い、データの広がりが最大になる方向を探す

という違いがあります。

## 6. 主成分得点

各データを主成分方向へ写した値を**主成分得点**と呼びます。

第1主成分得点は、

$$
PC1_i=\mathbf{x}_i\cdot\mathbf{u}_1
$$

として計算できます。

In [ ]:
pc1 = Xc @ u1
pc2 = Xc @ u2

plt.plot(pc1, "-o", label="PC1 score")
plt.plot(pc2, "-o", label="PC2 score")
plt.xlabel("Data number")
plt.ylabel("Principal component score")
plt.legend()
plt.show()

## 7. 寄与率

固有値は各主成分が持つ分散を表します。

そこで、

$$
\frac{\lambda_k}{\sum_j\lambda_j}
$$

を計算すると、全体のばらつきのうち第 $k$ 主成分がどの程度を表しているかが分かります。これを**寄与率**と呼びます。

In [ ]:
contribution = eigvals / eigvals.sum()

print("PC1 寄与率 =", contribution[0])
print("PC2 寄与率 =", contribution[1])

この例では、第1主成分だけでデータのばらつきの多くを表せることが分かります。

変数が2個ではなく数十個、数百個になっても基本的な考え方は同じです。海洋・気象データでは、多数の地点の水温や気圧などにPCAを適用し、空間的にまとまった変動を抽出することがあります。

## 8. まとめ

PCAでは、

1. 各変数から平均を引く
2. データがさまざまな方向にどれくらい広がっているかを考える
3. 最も大きく広がる方向を第1主成分とする
4. 共分散行列の固有値・固有ベクトルを使うと、その方向を計算できる
5. 固有値から寄与率を求められる

という流れでデータの主要な変動を取り出します。

第13回で扱った**分散・共分散・相関・最小二乗法**が、PCAを理解するための基礎になっています。